In [0]:
CREATE OR REPLACE VIEW airline_catalog.semantic.vw_airline_performance AS -- Crear o reemplazar la vista de desempeño de aerolíneas

WITH airline_metrics AS -- CTE para calcular métricas por aerolínea
(
    SELECT

        da.airline_code, -- Código de la aerolínea
        da.airline_name, -- Nombre de la aerolínea

        COUNT(*) AS total_flights, -- Total de vuelos

        SUM(CASE WHEN ff.flight_status = 'On Time' THEN 1 ELSE 0 END) AS on_time_flights, -- Vuelos a tiempo

        SUM(CASE WHEN ff.flight_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_flights, -- Vuelos retrasados

        SUM(CASE WHEN ff.cancelled THEN 1 ELSE 0 END) AS cancelled_flights, -- Vuelos cancelados

        ROUND(AVG(ff.departure_delay),2) AS avg_departure_delay, -- Promedio de retraso en salida

        ROUND(AVG(ff.arrival_delay),2) AS avg_arrival_delay, -- Promedio de retraso en llegada

        SUM(COALESCE(ff.departure_delay,0)) AS total_delay_minutes, -- Minutos totales de retraso en salida

        ROUND(AVG(ff.distance),2) AS avg_distance, -- Promedio de distancia recorrida

        ROUND(
            100.0 * SUM(CASE WHEN ff.flight_status = 'Delayed' THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS pct_delayed, -- Porcentaje de vuelos retrasados

        ROUND(
            100.0 * SUM(CASE WHEN ff.cancelled THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS pct_cancelled -- Porcentaje de vuelos cancelados

    FROM airline_catalog.gold.fact_flights ff -- Tabla de hechos de vuelos

    INNER JOIN airline_catalog.gold.dim_airline da -- Unión con dimensión de aerolínea
        ON ff.airline_id = da.airline_id -- Condición de unión por id de aerolínea

    GROUP BY
        da.airline_code, -- Agrupar por código de aerolínea
        da.airline_name -- Agrupar por nombre de aerolínea
)

SELECT

    *, -- Seleccionar todas las métricas calculadas

    RANK() OVER (
        ORDER BY avg_arrival_delay ASC
    ) AS punctuality_rank, -- Ranking de puntualidad (menor retraso en llegada)

    RANK() OVER (
        ORDER BY total_flights DESC
    ) AS traffic_rank, -- Ranking de tráfico (mayor cantidad de vuelos)

    DENSE_RANK() OVER (
        ORDER BY pct_cancelled ASC
    ) AS cancellation_rank -- Ranking de cancelación (menor porcentaje de cancelados)

FROM airline_metrics; -- Fuente: CTE de métricas por aerolínea

In [0]:
SELECT * FROM airline_catalog.semantic.vw_airline_performance	